# Stage 0 — MVP validation (run before any full sweep)

Runs the full pipeline (logit benchmarks · statistics · probing · figures) on three small pairs (Llama-3.2-3B · Qwen2.5-3B · Mistral-7B-v0.3) using `--limit 100` on the big benchmarks. Catches broken code, bad tokenizer interactions, and missing fields *before* you spend 40–55 H100-hours.

Expected runtime on a Colab A100/H100: **45–60 min**. Most of it is Mistral-7B.

This notebook is a thin viewer — the actual work lives in `scripts/run_mvp_validation.py` so it is testable, importable, and re-runnable from the command line.

In [ ]:
GITHUB_REPO = 'https://github.com/korneli777/llm-bias-eval.git'
DRIVE_DIR = '/content/drive/MyDrive/llm-bias-eval'

In [ ]:
import os, subprocess, shutil

if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive, userdata
    from huggingface_hub import login

    REPO_DIR = '/content/llm-bias-eval'
    drive.mount('/content/drive')
    os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)

    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull', '--ff-only'], check=True)

    results_link = os.path.join(REPO_DIR, 'results')
    if os.path.lexists(results_link):
        (os.unlink if os.path.islink(results_link) else shutil.rmtree)(results_link)
    os.symlink(f'{DRIVE_DIR}/results', results_link)

    subprocess.run(['pip', 'install', '-q', '-e', REPO_DIR], check=True)

    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['WANDB_PROJECT'] = 'llm-bias-eval'
    login(token=os.environ['HF_TOKEN'])

## Run the validation script

Writes JSONs to `results/mvp/`, figures to `figures/mvp/`, and the verdict report to `results/mvp/report.md`. Returns nonzero exit code on hard-fail — if the script complains, **do not start the full sweep**.

In [ ]:
!python scripts/run_mvp_validation.py --benchmark-limit 100

## Read the verdict

The report below is the single source of truth for whether to proceed. Look at the **Verdict** line first, then the **Hard fails** and **Soft observations** sections.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
report_path = Path('results/mvp/report.md')
display(Markdown(report_path.read_text()) if report_path.exists()
        else Markdown('**No report yet** — the script must finish first.'))

## Eyeball the figures

Sparse data (only 3 pairs) so visually they will look thin — the goal here is to verify each figure renders something sensible without crashing, not that it is publication-ready. Anything that looks broken, blank, or clipped is worth a flag.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
fig_dir = Path('figures/mvp')
if fig_dir.exists():
    for fp in sorted(fig_dir.glob('fig*.png')):
        print(fp.name)
        display(Image(filename=str(fp)))
else:
    print('No figures yet — the script must finish first.')